In [2]:
import torch
torch.cuda.is_available()
!pip install torch torchvision torchaudio
!pip install datasets transformers
!pip install scikit-learn tqdm pillow matplotlib

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models

from datasets import load_dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm import tqdm
import numpy as np

In [4]:
dataset = load_dataset("RohanRamesh/ff-images-dataset")

train_data = dataset["train"]
val_data   = dataset["validation"]
test_data  = dataset["test"]

print(len(train_data), len(val_data), len(test_data))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/474M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/318M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/178143 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21712 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24064 [00:00<?, ? examples/s]

178143 21712 24064


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
class DeepfakeDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]["image"]
        label = self.dataset[idx]["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
train_ds = DeepfakeDataset(train_data, train_transform)
val_ds   = DeepfakeDataset(val_data, val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(pretrained=True)

# Replace classifier
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 132MB/s] 


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [10]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [11]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)

            preds.extend(probs.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    preds = np.array(preds).ravel()
    targets = np.array(targets)

    acc = accuracy_score(targets, preds > 0.5)
    auc = roc_auc_score(targets, preds)

    return acc, auc


In [12]:
EPOCHS = 3

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_acc, val_auc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}")


100%|██████████| 5567/5567 [25:01<00:00,  3.71it/s]


Epoch 1
Train Loss: 0.1341
Val Acc: 0.9199, Val AUC: 0.9566


100%|██████████| 5567/5567 [25:07<00:00,  3.69it/s]


Epoch 2
Train Loss: 0.0605
Val Acc: 0.9126, Val AUC: 0.9543


100%|██████████| 5567/5567 [25:10<00:00,  3.69it/s]


Epoch 3
Train Loss: 0.0430
Val Acc: 0.9170, Val AUC: 0.9549


In [13]:
torch.save(model.state_dict(), "vigilant_eye.pth")